# Day 6 homework: Publish your agent (project)

The **course** packages the **DataTalks FAQ** agent in **`course/app/`**. Your homework is the **same architecture** for the agent you built in **Days 1–5** on **`pydantic/pydantic`** chunks: modular Python, **CLI**, **Streamlit**, optional **deploy**.

You will:

- Use the ready-made **`project/app/`** modules (or adapt them to your own repo/chunks)
- Run **`main.py`** and **`app.py`** locally
- Optionally **deploy** with `OPENAI_API_KEY` in Streamlit secrets

**Needs:** `OPENAI_API_KEY` (e.g. `ai_hero/.env`).


## 1. Clean layout (`project/app/`)

The runnable app lives next to your notebooks under **`project/app/`**:

| File | Role |
|------|------|
| `ingest.py` | Download repo zip, **chunk** markdown (`chunk` field), **`max_docs`** cap, **`Index.fit`** |
| `search_tools.py` | **`SearchTool`** → **`doc_search`** (same idea as notebook `doc_search`) |
| `search_agent.py` | **`Agent`** + prompt for Pydantic docs + GitHub **blob** links |
| `logs.py` | JSON logs under `LOGS_DIRECTORY` or `./logs` |
| `main.py` | **CLI**: index once, then `asyncio.run(agent.run(...))` |
| `app.py` | **Streamlit** chat + **streaming** via `run_stream_sync` + `stream_text` |

Design choices (aligned with your Day 4 homework):

- **Chunking** is on: sliding windows over `content` → rows stored with a **`chunk`** field for search.
- **`MAX_CHUNKS`** (default `400` in `main.py` / `app.py`) keeps startup fast; raise or set `None` in `ingest.index_data` if you index everything.
- The tool is exposed as **`doc_search`** → **`tools=[search_tool.doc_search]`** on the agent.


## 2. Install & run (local)

```bash
cd project/app
uv sync
```

CLI (downloads repo, builds chunks, then prompts):

```bash
uv run python main.py
```

Streamlit:

```bash
uv run streamlit run app.py
```

Open the printed URL. **First load** downloads and chunks the repo (can take a few minutes).

More detail: **`project/app/README.md`**.


## 3. Streaming (Pydantic AI ≥ 1.7)

Same pattern as **`course/app/app.py`**: **`agent.run_stream_sync(user_prompt=...)`** and **`result.stream_text(delta=True)`** for token-by-token output. When the stream completes, **`logs.log_interaction_to_file(agent, result.new_messages())`** writes one JSON file.

Optional: add **`message_history=...`** if you want the Streamlit session to pass prior turns into the agent (multi-turn memory).


## 4. Deploy (Streamlit Cloud)

1. Push your repo to GitHub.
2. Point the app at **`project/app`** with main file **`app.py`** (set the app **root** in the UI if your repo layout differs).
3. In **Secrets**:

   ```toml
   OPENAI_API_KEY = "sk-..."
   ```

4. Export dependencies if the host does not use **`uv`**:

   ```bash
   cd project/app
   uv export --no-dev --no-hashes -o requirements.txt
   ```

   Commit **`requirements.txt`** (already generated in this template when you run export).

5. If the filesystem is read-only, set **`LOGS_DIRECTORY`** (e.g. `/tmp/logs`) in secrets or env.

Public demos cost **API usage**—use a dedicated key and monitor spend.


## 5. Compare with the course app

| | Course `course/app/` | Homework `project/app/` |
|---|----------------------|-------------------------|
| Repo | `DataTalksClub/faq` | `pydantic/pydantic` |
| Rows | FAQ documents, **`data-engineering`** filter | **Chunked** markdown (`chunk` + metadata) |
| Index fields | `question`, `content`, `filename` | `chunk`, `title`, `description`, `filename` |
| Tool on agent | `search` | `doc_search` |
| Extra | — | **`MAX_CHUNKS`** limit |

Logging, Streamlit layout, and streaming mechanics match.


In [ ]:
# Quick sanity check: list project/app/*.py (run with cwd = project/ or adjust path)
from pathlib import Path

app_dir = Path("app")
if app_dir.is_dir():
    for p in sorted(app_dir.glob("*.py")):
        print(p)
else:
    print("Set the notebook working directory to the project/ folder, or open files in project/app/ in the editor.")


## Homework (deliverables)

- Walk through each file in **`project/app/`** and trace one request: **Streamlit → agent → `doc_search` → index → log file**.
- Confirm **CLI** and **Streamlit** both run with your `.env` key.
- Deploy to Streamlit Cloud **or** note why you skipped it (cost, keys, privacy).
- Optional: share a short post (see course Day 6) with your **app link** and repo.

[Course](https://alexeygrigorev.com/aihero/) · DataTalks **`#course-ai-hero`**
